![Databricks Academy](../Includes/images/db-academy.png)

# Demo - Connecting to Lakebase

There are a few approaches to connecting with Lakebase, which we'll illustrate in this demo using live notebook code.

In a subsequent demo, we'll apply these patterns to wire these connections into a Databricks app and handle end-to-end application integration.


## Learning Objectives

By the end of this demo, you will be able to:
- Explain the various options for accessing Lakebase, and when each is applicable
- Access Lakebase from Python code using a PostgreSQL driver
- Perform DDL and CRUD operations from Python code

## A. Overview of Lakebase

Lakebase is a fully managed PostgreSQL OLTP service integrated into Databricks, meant for low-latency transactional workloads (app state, feature serving, operational apps) that sit alongside your lakehouse analytics.

Lakebase features:

- Autoscaling compute based on demand
- Scale-to-zero when idle
- Organized resources as **projects** that support **branches** for isolation and restoring from any point in time
- Read replicas for scaling read-heavy workloads
- Can be configured to support high availability requirements

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 8px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">
  The original version, <strong>Lakebase Provisioned</strong>, used fixed-size, manually scaled database instances. All Provisioned instances have since been migrated to the current platform.
  </div>
</div>

## B. Classroom Setup (REQUIRED)

This section performs tasks required for the successful completion of this lab. Please run through all cells before proceeding.

### B1. Select Serverless Compute

Use serverless compute (v6) to run the cells within this notebook. Serverless will ensure that you're using the software environment on which this material was tested.

### B2. Run Setup

Run the following cell, which sets up your environment. This will:
- Configure variables used throughout the notebooks
- Create a Lakebase project for you to work with
- Output important configuration information below

In [0]:
%run ../Includes/Classroom-Setup-01.2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### B3. Understanding the Output

The output from the setup cell above provides a link to a pre-created Lakebase project for use with this demo. We'll be referring back to this page later in the demo.

## C. Connection Methods

Lakebase supports two main connection methods.

<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:20px;margin:20px 0;">
  <div style="background:#F9F7F4;border:1px solid #DCE0E2;border-radius:16px;box-shadow:0 2px 8px rgba(27,49,57,0.06);overflow:hidden;">
    <div style="height:8px;background:#FF5F46;"></div>
    <div style="padding:20px 22px;">
      <div style="font-size:24px;font-weight:700;line-height:1.2;color:#1B3139;margin-bottom:14px;">PostgreSQL Driver</div>
      <div style="font-size:15px;line-height:1.6;color:#303F47;margin-bottom:14px;">
      Provides native PostgreSQL access through an adapter like <code>psycopg3</code> or <code>SQLAlchemy</code>.
      </div>
    </div>
  </div>
  <div style="background:#F9F7F4;border:1px solid #DCE0E2;border-radius:16px;box-shadow:0 2px 8px rgba(27,49,57,0.06);overflow:hidden;">
    <div style="height:8px;background:#2272B4;"></div>
    <div style="padding:20px 22px;">
      <div style="font-size:24px;font-weight:700;line-height:1.2;color:#1B3139;margin-bottom:14px;">Lakebase Data API</div>
      <div style="font-size:15px;line-height:1.6;color:#303F47;margin-bottom:14px;">
        Provides HTTPS-based access to the database.
      </div>
    </div>
  </div>
</div>

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1100 500"
     font-family="system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif">
  <defs>
    <marker id="arrow" viewBox="0 0 12 12" refX="11" refY="6"
            markerWidth="10" markerHeight="10" orient="auto">
      <path d="M0,0 L12,6 L0,12 z" fill="#303F47"/>
    </marker>
  </defs>
  <rect width="1100" height="500" fill="#FFFFFF"/>
  <!-- =========================================================== -->
  <!-- Nested Lakebase containers (right side, bottom layer first)   -->
  <!-- =========================================================== -->
  <!-- Lakebase Project (outermost — Yellow 600) -->
  <rect x="600" y="50" width="460" height="410" rx="14" fill="#FFAB00"/>
  <text x="830" y="80" text-anchor="middle"
        font-size="14" font-weight="800" fill="#1B3139">Lakebase Project</text>
  <!-- Branch (middle — Oat Medium) -->
  <rect x="630" y="110" width="400" height="320" rx="12"
        fill="#EEEDE9" stroke="#DCE0E2" stroke-width="1"/>
  <text x="830" y="140" text-anchor="middle"
        font-size="14" font-weight="800" fill="#1B3139">Branch</text>
  <!-- Lakebase Endpoint (inner — Oat Light) -->
  <rect x="660" y="165" width="340" height="240" rx="10"
        fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1"/>
  <text x="830" y="195" text-anchor="middle"
        font-size="14" font-weight="800" fill="#1B3139">Compute</text>
  <!-- Database (innermost — white) -->
  <rect x="760" y="245" width="140" height="80" rx="8"
        fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.2"/>
  <text x="830" y="293" text-anchor="middle"
        font-size="16" font-weight="800" fill="#1B3139">Database</text>
  <!-- =========================================================== -->
  <!-- Left-side boxes                                               -->
  <!-- =========================================================== -->
  <!-- Python Clients -->
  <rect x="40" y="120" width="120" height="80" rx="12"
        fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1.2"/>
  <text x="100" y="155" text-anchor="middle"
        font-size="14" font-weight="700" fill="#1B3139">Python-based</text>
  <text x="100" y="178" text-anchor="middle"
        font-size="14" font-weight="700" fill="#1B3139">clients</text>
  <!-- PostgreSQL Driver (Lava 600) -->
  <rect x="220" y="120" width="160" height="80" rx="12" fill="#FF3621"/>
  <text x="300" y="155" text-anchor="middle"
        font-size="15" font-weight="800" fill="#FFFFFF">PostgreSQL</text>
  <text x="300" y="178" text-anchor="middle"
        font-size="15" font-weight="800" fill="#FFFFFF">Driver</text>
  <!-- HTTP Clients -->
  <rect x="40" y="300" width="120" height="80" rx="12"
        fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1.2"/>
  <text x="100" y="335" text-anchor="middle"
        font-size="14" font-weight="700" fill="#1B3139">HTTP</text>
  <text x="100" y="358" text-anchor="middle"
        font-size="14" font-weight="700" fill="#1B3139">clients</text>
  <!-- Lakebase Data API (Blue 600) -->
  <rect x="220" y="300" width="160" height="80" rx="12" fill="#2272B4"/>
  <text x="300" y="335" text-anchor="middle"
        font-size="15" font-weight="800" fill="#FFFFFF">Lakebase</text>
  <text x="300" y="358" text-anchor="middle"
        font-size="15" font-weight="800" fill="#FFFFFF">Data API</text>
  <!-- =========================================================== -->
  <!-- Arrows (on top of all boxes)                                  -->
  <!-- =========================================================== -->
  <!-- Python Clients → PostgreSQL Driver -->
  <line x1="160" y1="160" x2="220" y2="160" stroke="#303F47" stroke-width="2"
        marker-end="url(#arrow)"/>
  <!-- HTTP Clients → Lakebase Data API -->
  <line x1="160" y1="340" x2="220" y2="340" stroke="#303F47" stroke-width="2"
        marker-end="url(#arrow)"/>
  <!-- Python Clients → Lakebase Data API (NEW — Python clients can use either) -->
  <path d="M 130 200 C 130 250, 200 300, 220 320"
        fill="none" stroke="#303F47" stroke-width="2"
        marker-end="url(#arrow)"/>
  <!-- PostgreSQL Driver → Database (upper entry, now angled) -->
  <path d="M 380 160 L 600 160 C 680 160, 730 220, 760 250"
        fill="none" stroke="#303F47" stroke-width="2"
        marker-end="url(#arrow)"/>
  <!-- Lakebase Data API → Database (lower entry, now angled) -->
  <path d="M 380 340 L 600 340 C 680 340, 730 350, 760 320"
        fill="none" stroke="#303F47" stroke-width="2"
        marker-end="url(#arrow)"/>
  <!-- =========================================================== -->
  <!-- Pill labels on top of arrows                                  -->
  <!-- =========================================================== -->
  <!-- PostgreSQL wire protocol -->
  <rect x="410" y="146" width="170" height="28" rx="10"
        fill="#EEEDE9" stroke="#DCE0E2" stroke-width="1"/>
  <text x="495" y="165" text-anchor="middle"
        font-size="12" font-weight="600" fill="#1B3139">PostgreSQL wire protocol</text>
  <!-- HTTPS -->
  <rect x="455" y="326" width="70" height="28" rx="10"
        fill="#EEEDE9" stroke="#DCE0E2" stroke-width="1"/>
  <text x="490" y="345" text-anchor="middle"
        font-size="12" font-weight="600" fill="#1B3139">HTTPS</text>
</svg>

### C1. PostgreSQL Driver

Drivers like [psycopg3](https://www.psycopg.org/) or [SQLAlchemy](https://www.sqlalchemy.org/) can be employed to directly connect to Lakebase in the same way as any other Postgres database. In this case, your code uses the standard PostgreSQL wire protocol to interact with the database server.

This is generally the best fit for Databricks apps, or for any situation requiring:
- close-to-DB interactions
- access to the entirety of SQL (complex queries, multi-statement transactions, prepared statements, and more) and/or Postgres ecosystem
- other libraries that expect a DB-API driver
- high-throughput, low-latency calls

### C2. Lakebase Data API

Lakebase can also expose a [PostgREST](https://docs.postgrest.org/en/stable/references/api.html)-compatible REST API [[AWS](https://docs.databricks.com/aws/en/oltp/projects/data-api) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/projects/data-api) | [GCP](https://docs.databricks.com/gcp/en/oltp/projects/data-api)] that sits in front of the database. Schemas, tables and views are exposed as HTTP resources, with CRUD operations via standard HTTP verbs (`GET`/`POST`/`PATCH`/`DELETE`) and rich query parameters for filtering, ordering, pagination, and joins.

This is a good option for:
- any client where using a Python-based PostgreSQL driver is not suitable or possible
- serverless functions or edge runtimes that prefer stateless HTTP over maintaining connections
- simple CRUD and query patterns

### C3. Comparison

Here is a summary of how the two connectivity options compare:

<table>
  <thead>
    <tr>
      <th>Aspect</th>
      <th>Lakebase Data API</th>
      <th>PostgreSQL driver (psycopg3, SQLAlchemy, etc)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Protocol</td>
      <td>REST (HTTPS + JSON)</td>
      <td>PostgreSQL wire protocol</td>
    </tr>
    <tr>
      <td>Authorization model</td>
      <td>Databricks OAuth bearer token in Authorization header</td>
      <td>Short-lived DB credentials (token as password) or Postgres passwords</td>
    </tr>
    <tr>
      <td>API surface</td>
      <td>Auto-generated CRUD endpoints; query via URL params</td>
      <td>Arbitrary SQL, transactions, stored procedures, driver features</td>
    </tr>
    <tr>
      <td>Shape of responses</td>
      <td>JSON documents</td>
      <td>Python DB-API rows / typed values</td>
    </tr>
    <tr>
      <td>Connection behavior</td>
      <td>Stateless, per-request HTTP</td>
      <td>Stateful connections, pooling, prepared statements</td>
    </tr>
    <tr>
      <td>Usage</td>
      <td>Web/mobile/serverless, third-party integrations, simple CRUD</td>
      <td>Backend services, ETL, complex queries, ORMs, high-throughput servers</td>
    </tr>
    <tr>
      <td>Language/tooling neutrality</td>
      <td>Very high (any HTTP client)</td>
      <td>Python-specific (but same pattern in other languages via other drivers)</td>
    </tr>
    <tr>
      <td>Performance/throughput</td>
      <td>Good</td>
      <td>Best</td>
    </tr>
  </tbody>
</table>
<style>
table{
  border-collapse: separate;
  border-spacing: 0;
  width: auto;
  max-width: 100%;
  margin: 16px 0;
  float: none;
  clear: both;
  display: table;
  font-size: 0.95em;
  line-height: 1.35;
  background: #ffffff;
  border-radius: 8px;
  overflow: hidden;
  box-shadow: 0 0 0 1px #303F47;
  margin-left: 2px !important;
  margin-right: 2px !important;
  box-sizing: border-box !important;
}
thead th{
  background: #DADADA;
  color: #1B3139;
  font-weight: 600;
  text-align: left;
  padding: 10px 12px;
}
tbody td{
  padding: 10px 12px;
  border-top: 1px solid #DCE0E2;
}
tbody tr:nth-child(odd){
  background: #FFFFFF !important;
}
tbody tr:nth-child(even){
  background: #F5F3F2 !important;
}
</style>

## D. Using a PostgreSQL Driver

In this section, we'll see how to connect to Lakebase using a PostgreSQL driver. In this example we'll use [psycopg3](https://www.psycopg.org/psycopg3/docs/), a modern, widely-used PostgreSQL driver for Python. [SQLAlchemy](https://www.sqlalchemy.org/) is another good option.

### D1. PostgreSQL Connection Overview

In the simplest case, a Python program connects to Postgres by opening a single connection, running queries, and closing the connection when it is finished. This pattern works well for scripts or notebooks where a small amount of work is performed sequentially. However, web applications often execute many queries across multiple concurrent users or workers. In these environments, repeatedly opening and closing database connections becomes inefficient and can limit throughput.

To solve this problem, we turn to connection pools. In the remainder of the section, we highlight a pattern to employ connection pools to connect to Lakebase.

<div style="display:block; margin:18px auto; max-width:760px; width:100%; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; color:#1B3139;">
  <svg viewBox="0 0 760 380" width="100%" style="display:block;">
    <defs>
      <marker id="pool-arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto">
        <path d="M0,0 L0,6 L8,3 z" fill="#5B6B73"></path>
      </marker>
      <filter id="pool-shadow" x="-20%" y="-20%" width="140%" height="140%">
        <feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#1B3139" flood-opacity="0.08"></feDropShadow>
      </filter>
    </defs>
    <!-- Connection pool container -->
    <rect x="260" y="30" width="210" height="320" rx="16"
          fill="#F9F7F4" stroke="#5B6B73" stroke-width="1.6" filter="url(#pool-shadow)"></rect>
    <text x="365" y="58" text-anchor="middle" font-size="17" font-weight="800" fill="#1B3139">Connection pool</text>
    <!-- Connections (evenly spaced) -->
    <!-- centers: ~110, 190, 270 -->
    <rect x="286" y="80" width="158" height="60" rx="10"
          fill="#FFFFFF" stroke="#5B6B73" stroke-width="1.5"></rect>
    <text x="365" y="115" text-anchor="middle" font-size="16" font-weight="700">Connection</text>
    <rect x="286" y="160" width="158" height="60" rx="10"
          fill="#FFFFFF" stroke="#5B6B73" stroke-width="1.5"></rect>
    <text x="365" y="195" text-anchor="middle" font-size="16" font-weight="700">Connection</text>
    <rect x="286" y="240" width="158" height="60" rx="10"
          fill="#FFFFFF" stroke="#5B6B73" stroke-width="1.5" opacity="0.65"></rect>
    <text x="365" y="275" text-anchor="middle" font-size="15" fill="#5B6B73">Connection</text>
    <!-- Clients aligned to connections -->
    <rect x="36" y="80" width="110" height="60" rx="10"
          fill="#FF5F46" stroke="#98102A" filter="url(#pool-shadow)"></rect>
    <text x="91" y="115" text-anchor="middle" font-size="16" font-weight="800" fill="#FFFFFF">Client</text>
    <rect x="36" y="160" width="110" height="60" rx="10"
          fill="#FF5F46" stroke="#98102A" filter="url(#pool-shadow)"></rect>
    <text x="91" y="195" text-anchor="middle" font-size="16" font-weight="800" fill="#FFFFFF">Client</text>
    <!-- Database -->
    <rect x="610" y="130" width="120" height="100" rx="12"
          fill="#1F77B4" stroke="#1B3139" filter="url(#pool-shadow)"></rect>
    <text x="670" y="170" text-anchor="middle" font-size="16" font-weight="800" fill="#FFFFFF">Postgres</text>
    <text x="670" y="195" text-anchor="middle" font-size="16" font-weight="800" fill="#FFFFFF">Database</text>
    <!-- Checkout arrows (aligned cleanly) -->
    <line x1="146" y1="110" x2="286" y2="110"
          stroke="#5B6B73" stroke-width="2.2" marker-end="url(#pool-arrow)"></line>
    <rect x="165" y="94" width="76" height="28" rx="6" fill="#EEEDE9"></rect>
    <text x="203" y="113" text-anchor="middle" font-size="15">checkout</text>
    <line x1="146" y1="190" x2="286" y2="190"
          stroke="#5B6B73" stroke-width="2.2" marker-end="url(#pool-arrow)"></line>
    <rect x="165" y="174" width="76" height="28" rx="6" fill="#EEEDE9"></rect>
    <text x="203" y="193" text-anchor="middle" font-size="15">checkout</text>
    <!-- Transactions (still slightly curved for life) -->
    <path d="M444 110 C520 110, 550 130, 610 155"
          fill="none" stroke="#5B6B73" stroke-width="2.2" marker-end="url(#pool-arrow)"></path>
    <rect x="500" y="95" width="100" height="28" rx="6" fill="#EEEDE9"></rect>
    <text x="550" y="114" text-anchor="middle" font-size="15">transaction</text>
    <path d="M444 190 C520 190, 550 180, 610 175"
          fill="none" stroke="#5B6B73" stroke-width="2.2" marker-end="url(#pool-arrow)"></path>
    <rect x="500" y="170" width="100" height="28" rx="6" fill="#EEEDE9"></rect>
    <text x="550" y="189" text-anchor="middle" font-size="15">transaction</text>
  </svg>
</div>

### D2. Establishing a Connection Pool

A **connection pool** is a managed collection of open database connections that your code checks out, uses for a query or transaction, and then returns to the pool instead of opening and closing new connections each time. This avoids the overhead of repeatedly establishing Postgres connections, which is especially important for Databricks apps that serve many short requests or run concurrently across multiple workers.

Pools are also a natural place to integrate credential rotation. In the example below, the call to `postgres.generate_database_credential()` ensures that each new pooled connection receives a fresh short-lived token.

By centralizing connection lifecycle and token handling in the pool, your app code stays simple, you minimize latency per request, and you naturally scale to higher QPS without running into connection‑setup bottlenecks.

<div style="display:block; margin:18px auto; max-width:980px; width:100%; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; color:#1B3139;">
  <svg viewBox="0 0 980 470" width="100%" style="display:block;">
    <defs>
      <marker id="seq-arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto">
        <path d="M0,0 L0,6 L8,3 z" fill="#5B6B73"></path>
      </marker>
      <filter id="seq-shadow" x="-20%" y="-20%" width="140%" height="140%">
        <feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#1B3139" flood-opacity="0.07"></feDropShadow>
      </filter>
    </defs>
    <!-- Participant cards -->
    <rect x="24" y="20" width="150" height="66" rx="10" fill="#F9F7F4" stroke="#DCE0E2" filter="url(#seq-shadow)"></rect>
    <text x="99" y="58" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Client</text>
    <rect x="265" y="20" width="180" height="66" rx="10" fill="#FFF4F2" stroke="#FFD2CB" filter="url(#seq-shadow)"></rect>
    <text x="355" y="49" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Connection Pool</text>
    <text x="355" y="68" text-anchor="middle" font-size="14" fill="#5B6B73">(psycopg3)</text>
    <rect x="585" y="20" width="150" height="66" rx="10" fill="#EEF6FF" stroke="#CFE4FF" filter="url(#seq-shadow)"></rect>
    <text x="660" y="58" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Databricks</text>
    <rect x="815" y="20" width="150" height="66" rx="10" fill="#F4FBF7" stroke="#CFE9D8" filter="url(#seq-shadow)"></rect>
    <text x="890" y="49" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Postgres</text>
    <text x="890" y="68" text-anchor="middle" font-size="14" fill="#5B6B73">(Lakebase)</text>
    <!-- Bottom participant cards -->
    <rect x="24" y="382" width="150" height="66" rx="10" fill="#F9F7F4" stroke="#DCE0E2"></rect>
    <text x="99" y="421" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Client</text>
    <rect x="265" y="382" width="180" height="66" rx="10" fill="#FFF4F2" stroke="#FFD2CB"></rect>
    <text x="355" y="411" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Connection Pool</text>
    <text x="355" y="430" text-anchor="middle" font-size="14" fill="#5B6B73">(psycopg3)</text>
    <rect x="585" y="382" width="150" height="66" rx="10" fill="#EEF6FF" stroke="#CFE4FF"></rect>
    <text x="660" y="421" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Databricks</text>
    <rect x="815" y="382" width="150" height="66" rx="10" fill="#F4FBF7" stroke="#CFE9D8"></rect>
    <text x="890" y="411" text-anchor="middle" font-size="16" font-weight="700" fill="#1B3139">Postgres</text>
    <text x="890" y="430" text-anchor="middle" font-size="14" fill="#5B6B73">(Lakebase)</text>
    <!-- Lifelines -->
    <line x1="99" y1="86" x2="99" y2="382" stroke="#D0D6DA" stroke-width="3"></line>
    <line x1="355" y1="86" x2="355" y2="382" stroke="#FFD2CB" stroke-width="2"></line>
    <line x1="660" y1="86" x2="660" y2="382" stroke="#CFE4FF" stroke-width="2"></line>
    <line x1="890" y1="86" x2="890" y2="382" stroke="#CFE9D8" stroke-width="2"></line>
    <!-- Activations -->
    <line x1="355" y1="130" x2="355" y2="178" stroke="#FFD2CB" stroke-width="6"></line>
    <line x1="355" y1="226" x2="355" y2="274" stroke="#FFD2CB" stroke-width="6"></line>
    <line x1="355" y1="322" x2="355" y2="358" stroke="#FFD2CB" stroke-width="6"></line>
    <line x1="660" y1="178" x2="660" y2="226" stroke="#CFE4FF" stroke-width="6"></line>
    <line x1="890" y1="274" x2="890" y2="322" stroke="#CFE9D8" stroke-width="6"></line>
    <!-- Messages -->
    <!-- Client -> Pool -->
    <line x1="99" y1="130" x2="355" y2="130" stroke="#5B6B73" stroke-width="2.4" marker-end="url(#seq-arrow)"></line>
    <text x="225" y="116" text-anchor="middle" font-size="15" fill="#1B3139">Request connection</text>
    <!-- Pool -> Databricks -->
    <line x1="355" y1="178" x2="660" y2="178" stroke="#5B6B73" stroke-width="2.4" marker-end="url(#seq-arrow)"></line>
    <text x="507" y="163" text-anchor="middle" font-size="15" fill="#1B3139">generate_database_credential()</text>
    <!-- Databricks -> Pool token -->
    <line x1="660" y1="226" x2="355" y2="226" stroke="#5B6B73" stroke-width="2.2" stroke-dasharray="4 4" marker-end="url(#seq-arrow)"></line>
    <text x="508" y="212" text-anchor="middle" font-size="15" fill="#1B3139">Token</text>
    <!-- Pool -> Postgres -->
    <line x1="355" y1="274" x2="890" y2="274" stroke="#5B6B73" stroke-width="2.4" marker-end="url(#seq-arrow)"></line>
    <text x="620" y="259" text-anchor="middle" font-size="15" fill="#1B3139">connect(password=token)</text>
    <!-- Postgres -> Pool -->
    <line x1="890" y1="322" x2="355" y2="322" stroke="#5B6B73" stroke-width="2.2" stroke-dasharray="4 4" marker-end="url(#seq-arrow)"></line>
    <text x="620" y="307" text-anchor="middle" font-size="15" fill="#1B3139">Connection established</text>
    <!-- Pool -> Client -->
    <line x1="355" y1="358" x2="99" y2="358" stroke="#5B6B73" stroke-width="2.2" stroke-dasharray="4 4" marker-end="url(#seq-arrow)"></line>
    <text x="225" y="344" text-anchor="middle" font-size="15" fill="#1B3139">New pooled connection</text>
  </svg>
</div>

Run the following cell, which defines a `Connection` subclass that injects authentication token management. This pattern is considered best practice when using **psycopg3** to connect to Lakebase.

In [0]:
import os

os.environ["PSYCOPG_IMPL"] = "python"

import psycopg
from psycopg_pool import ConnectionPool

class LakebaseConnection(psycopg.Connection):

    @classmethod
    def connect(
        cls,
        conninfo: str="",
        *,
        workspace=None,
        endpoint=None,
        **kwargs
    ):
        # fetch new Lakebase token
        cred = workspace.postgres.generate_database_credential(endpoint=endpoint)

        # inject token into the connect() call
        kwargs["password"] = cred.token
        conn = super().connect(conninfo, **kwargs)

        return conn

### D3. Instantiating the Pool

Next, we instantiate a pool using the connection class above, and a connection string we construct from:
- the database name (in this example, we'll assume `databricks_postgres`)
- the username (in this example, we'll assume the user running this notebook)
- the host, which we obtain from the project that was set up at the beginning of this notebook, using the `postgres.get_endpoint()` API

In [0]:
# assume by default
database = 'databricks_postgres'

# use currently authenticated user
username = workspace.current_user.me().user_name

# obtain host from Lakebase compute (coming from setup cell above)
host = workspace.postgres.get_endpoint(name=my_lakebase_compute).status.hosts.host

# initialize connection pool using the Lakebase connection handler
pool = ConnectionPool(
    conninfo=f"dbname={database} user={username} host={host} port=5432 sslmode=require",
    connection_class=LakebaseConnection,
    kwargs= {
        # pass workspace connection and Lakebase compute to the connect() method
        "workspace": workspace,
        "endpoint": my_lakebase_compute
    },
    min_size=1,
    max_size=10,
    open=True
)

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-5716406749851139>, line 8
      5 username = workspace.current_user.me().user_name
      7 # obtain host from Lakebase compute (coming from setup cell above)
----> 8 host = workspace.postgres.get_endpoint(name=my_lakebase_compute).status.hosts.host
     10 # initialize connection pool using the Lakebase connection handler
     11 pool = ConnectionPool(
     12     conninfo=f"dbname={database} user={username} host={host} port=5432 sslmode=require",
     13     connection_class=LakebaseConnection,
   (...)
     21     open=True
     22 )

AttributeError: 'WorkspaceClient' object has no attribute 'postgres'

### D4. Executing SQL

With a connection pool ready, we can now illustrate how to grab a connection and run SQL statements that:
1. Create a schema
1. Create a table in that schema

In [0]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        # create a schema named "test"
        cur.execute("CREATE SCHEMA IF NOT EXISTS test")

        # create a small table named "hello"
        cur.execute("""
            CREATE TABLE IF NOT EXISTS test.hello AS
            SELECT
                1 AS id,
                'Hello Lakebase' AS message,
                current_user AS created_by,
                current_timestamp AS created_at
        """)

    conn.commit()

### D5. Validating Results

Let's explore how to validate the current state of the database using SQL and the Databricks UI.

#### D5.1 Executing a Read Operation in SQL

The code below validates the work we did above by displaying the contents of the table. It also illustrates the pattern for table reading using the PostgreSQL driver:

In [0]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        # read contents of table "hello"
        cur.execute("SELECT * FROM test.hello")
        print(cur.fetchall())

#### D5.2 Inspecting the Database in the UI

We can also inspect tables in the UI. In the Lakebase project page (which we opened in the first section of this notebook):
1. Select **Tables** in the left sidebar
1. Use the schema dropdown to select `test`
1. Select the **hello** table and note the contents

### D6. Recap

In this section, we connected to Lakebase using the **psycopg3** driver. This driver provides modern capabilities for doing complex, high-bandwidth interactions with Postgres. Furthermore, the patterns we saw align with best practices for maintaining fresh tokens in a connection pool to ensure performant behavior.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 8px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Important Note
  </strong>
  <div style="color:#333;">
  These connections worked out-of-the-box because we connected using the same identity that was used to create the project; so we automatically had a role enabled that allowed us to connect to and create schemas.
  
  When accessing from a Databricks app, we will have to perform some additional configuration to allow the app service principal to access the database. We will cover this in a separate lab.
  </div>
</div>

## E. Cleanup

Please run the following cell to clean up resources before proceeding.

In [0]:
cleanup()

## F. Conclusion
In this demo, we:
- **Covered what Lakebase provides**: elastic compute with scale-to-zero, organized as projects with branches and read replicas.
- **Connected using a Python PostgreSQL driver**: we used `psycopg3` with connection pools to ensure high performance with fresh, short-lived tokens that are automatically rotated, following the recommended pattern for production apps.
- **Performed and verified transactions**: from Python, we executed statements to:
   - create a schema
   - create a table
   - insert data into a table
   - read data from a table

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>